# Full Cross-Dataset Benchmark — FactorizePhys_FSAM_Res & PhysFormer

Evaluates 10 model configurations across 3 datasets:
- **UBFC-rPPG** (42 subjects): FactorizePhys results copied from notebook 06; PhysFormer run fresh.
- **rPPG-10** (up to 27 subjects): all 10 configs run fresh.
- **MCD-rPPG** (last 100 patient_ids, IriunWebcam only): all 10 configs run fresh.

**FactorizePhys configs** (FSAM_Res architecture):
1. SCAMPS alone
2. PURE alone
3. iBVP alone
4. Soup: SCAMPS+PURE
5. Soup: SCAMPS+iBVP
6. Soup: SCAMPS+PURE+iBVP

**PhysFormer configs** (ViT_ST_ST_Compact3_TDC_gra_sharp):
7. SCAMPS alone
8. PURE alone
9. iBVP alone
10. Soup: SCAMPS+PURE+iBVP

**Note on Pearson r**: All Pearson r values here are HR-level correlations across subjects
(predicted HR vs GT HR), NOT waveform-level correlation.
Papers typically report waveform-level r; our values are not directly comparable to paper r values.

In [1]:
# Cell 1 — Imports and config
import sys, time, warnings, json, gc
from pathlib import Path
from collections import OrderedDict, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import cv2
from torch.utils.data import Dataset, DataLoader
from scipy.signal import butter, filtfilt
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('..').resolve()
FP_ROOT      = PROJECT_ROOT / 'external' / 'FactorizePhys'
CKPT_DIR     = FP_ROOT / 'final_model_release'

# FactorizePhys checkpoints
CKPT_FP_SCAMPS = CKPT_DIR / 'SCAMPS_FactorizePhys_FSAM_Res.pth'
CKPT_FP_PURE   = CKPT_DIR / 'PURE_FactorizePhys_FSAM_Res.pth'
CKPT_FP_IBVP   = CKPT_DIR / 'iBVP_FactorizePhys_FSAM_Res.pth'

# PhysFormer checkpoints
CKPT_PF_SCAMPS = CKPT_DIR / 'SCAMPS_PhysFormer.pth'
CKPT_PF_PURE   = CKPT_DIR / 'PURE_PhysFormer.pth'
CKPT_PF_IBVP   = CKPT_DIR / 'iBVP_PhysFormer.pth'

# Dataset paths
UBFC_CACHE    = Path('/tmp/ubfc_y5f_clips_72.pt')
RPPG10_ROOT   = PROJECT_ROOT / 'rppg_dataset' / 'rPPG-10' / 'Dataset_rPPG-10'
MCD_ROOT      = PROJECT_ROOT / 'rppg_dataset' / 'MCD-rPPG'
MCD_DB_CSV    = MCD_ROOT / 'db.csv'
MCD_VIDEO_DIR = MCD_ROOT / 'video'

# Output paths
BEST_CKPT_DIR = PROJECT_ROOT / 'checkpoints' / 'best_pretrained'
BEST_CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Eval settings
BATCH_SIZE    = 4      # conservative for PhysFormer memory
RPPG10_CLIPS  = 160   # frames per clip
MCD_N_FRAMES  = 960   # load first 960 frames per video
MCD_CLIP_LEN  = 160   # frames per clip -> 6 clips per video
FPS           = 30.0

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Add FactorizePhys to sys.path for model imports
if str(FP_ROOT) not in sys.path:
    sys.path.insert(0, str(FP_ROOT))

print(f'Device          : {device}')
print(f'FP SCAMPS ckpt  : {CKPT_FP_SCAMPS.exists()}')
print(f'FP PURE ckpt    : {CKPT_FP_PURE.exists()}')
print(f'FP iBVP ckpt    : {CKPT_FP_IBVP.exists()}')
print(f'PF SCAMPS ckpt  : {CKPT_PF_SCAMPS.exists()}')
print(f'PF PURE ckpt    : {CKPT_PF_PURE.exists()}')
print(f'PF iBVP ckpt    : {CKPT_PF_IBVP.exists()}')
print(f'UBFC cache      : {UBFC_CACHE.exists()}')
print(f'rPPG-10 root    : {RPPG10_ROOT.exists()}')
print(f'MCD-rPPG root   : {MCD_ROOT.exists()}')

Device          : cuda:0
FP SCAMPS ckpt  : True
FP PURE ckpt    : True
FP iBVP ckpt    : True
PF SCAMPS ckpt  : True
PF PURE ckpt    : True
PF iBVP ckpt    : True
UBFC cache      : True
rPPG-10 root    : True
MCD-rPPG root   : True


In [2]:
# Cell 2 — ClearML init
try:
    from clearml import Task
    task = Task.init(
        project_name='rPPG',
        task_name='Full Benchmark',
        reuse_last_task_id=False,
    )
    task.set_parameter('datasets',       'UBFC-rPPG, rPPG-10, MCD-rPPG')
    task.set_parameter('fp_ckpt_dir',    str(CKPT_DIR))
    task.set_parameter('ubfc_cache',     str(UBFC_CACHE))
    task.set_parameter('rppg10_root',    str(RPPG10_ROOT))
    task.set_parameter('mcd_root',       str(MCD_ROOT))
    task.set_parameter('batch_size',     BATCH_SIZE)
    task.set_parameter('mcd_n_frames',   MCD_N_FRAMES)
    task.set_parameter('mcd_clip_len',   MCD_CLIP_LEN)
    task.set_parameter('device',         str(device))
    task.set_parameter('fps',            FPS)
    logger = task.get_logger()
    print('ClearML task initialized:', task.id)
except Exception as e:
    logger = None
    task   = None
    print(f'ClearML unavailable — local-only mode. ({e})')

ClearML Task: created new task id=5dac48d734a141dd9aa32ffc43a634cd


2026-05-23 19:44:29,233 - clearml.Repository Detection - WARNING - Failed accessing the jupyter server(s): []


ClearML results page: https://app.clear.ml/projects/288b5746cdc7439ea1151b913b846646/experiments/5dac48d734a141dd9aa32ffc43a634cd/output/log


ClearML task initialized: 5dac48d734a141dd9aa32ffc43a634cd


In [3]:
# Cell 3 — Signal processing utilities (HR extraction)

def extract_hr(waveform, fps=30.0, low_bpm=36, high_bpm=198):
    """Extract HR from a 1-D waveform via bandpass + FFT peak.
    Identical to notebook 05/06 protocol."""
    lo, hi = low_bpm / 60.0, high_bpm / 60.0
    nyq    = fps / 2.0
    N      = len(waveform)
    if N < 8:
        return 75.0
    try:
        b, a = butter(4, [lo / nyq, hi / nyq], btype='bandpass')
        sig  = filtfilt(b, a, waveform.astype('float64'))
    except Exception:
        sig  = waveform.astype('float64')
    window = np.hanning(N)
    fft    = np.abs(np.fft.rfft(sig * window))
    freq   = np.fft.rfftfreq(N, d=1.0 / fps)
    mask   = (freq >= lo) & (freq <= hi)
    if not mask.any():
        return 75.0
    return float(freq[mask][np.argmax(fft[mask])] * 60.0)


def extract_hr_ecg(ecg_signal, ecg_fs=1000.0):
    """Extract HR from raw ECG via R-peak detection (Pan-Tompkins style).
    FFT is unreliable for ECG (non-sinusoidal). R-peak detection is standard."""
    from scipy.signal import find_peaks
    nyq = ecg_fs / 2.0
    b, a = butter(4, [5.0 / nyq, 20.0 / nyq], btype='bandpass')
    filtered = filtfilt(b, a, ecg_signal.astype('float64'))
    squared  = filtered ** 2
    min_dist = int(0.3 * ecg_fs)  # 300ms = max 200 bpm
    threshold = np.percentile(squared, 75)
    peaks, _  = find_peaks(squared, distance=min_dist, height=threshold)
    if len(peaks) < 2:
        return 75.0
    rr = np.diff(peaks) / ecg_fs  # RR intervals in seconds
    rr = rr[(rr >= 0.3) & (rr <= 2.0)]  # keep 30-200 bpm range
    if len(rr) == 0:
        return 75.0
    return float(60.0 / np.mean(rr))


def compute_metrics(pred_hrs, gt_hrs):
    """Compute MAE, RMSE, Pearson r from parallel arrays of per-subject HRs."""
    err  = np.array(pred_hrs) - np.array(gt_hrs)
    mae  = float(np.abs(err).mean())
    rmse = float(np.sqrt((err ** 2).mean()))
    if len(pred_hrs) > 1:
        r = float(np.corrcoef(pred_hrs, gt_hrs)[0, 1])
    else:
        r = float('nan')
    return {'MAE': mae, 'RMSE': rmse, 'Pearson_r': r}


print('Signal processing utilities ready.')

Signal processing utilities ready.


In [4]:
# Cell 4 — Model architecture utilities

from yacs.config import CfgNode as CN
from neural_methods.model.FactorizePhys.FactorizePhys import FactorizePhys
from neural_methods.model.PhysFormer import ViT_ST_ST_Compact3_TDC_gra_sharp


def make_fp_cfg():
    """FactorizePhys FSAM_Res config."""
    cfg = CN()
    cfg.CHANNELS     = 3
    cfg.FRAME_NUM    = 160
    cfg.MD_FSAM      = True
    cfg.MD_TYPE      = 'NMF'
    cfg.MD_R         = 1
    cfg.MD_S         = 1
    cfg.MD_STEPS     = 4
    cfg.MD_RESIDUAL  = True
    cfg.MD_INFERENCE = True
    return cfg


def load_stripped(ckpt_path):
    """Load checkpoint and strip 'module.' prefix from DataParallel wrapping."""
    raw = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
    return OrderedDict((k.replace('module.', ''), v) for k, v in raw.items())


def make_soup(ckpt_paths):
    """Equal-weight arithmetic mean of state_dicts from a list of checkpoint paths."""
    dicts = [load_stripped(p) for p in ckpt_paths]
    avg   = OrderedDict()
    for key in dicts[0]:
        avg[key] = torch.stack([d[key].float() for d in dicts]).mean(dim=0)
    return avg


def build_factorizephys(state_dict, device):
    model = FactorizePhys(
        frames=160, md_config=make_fp_cfg(), device=device, in_channels=3
    ).to(device)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f'  [FP] Missing keys   : {len(missing)}')
    if unexpected:
        print(f'  [FP] Unexpected keys: {len(unexpected)}')
    model.eval()
    return model


def build_physformer(state_dict, device):
    model = ViT_ST_ST_Compact3_TDC_gra_sharp(
        image_size=(160, 72, 72),
        patches=(4, 4, 4),
        dim=96,
        ff_dim=144,
        num_heads=4,
        num_layers=12,
        dropout_rate=0.1,
        theta=0.7,
    ).to(device)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f'  [PF] Missing keys   : {len(missing)}')
    if unexpected:
        print(f'  [PF] Unexpected keys: {len(unexpected)}')
    model.eval()
    return model


# Parameter counts
_fp = FactorizePhys(frames=160, md_config=make_fp_cfg(), device='cpu', in_channels=3)
_pf = ViT_ST_ST_Compact3_TDC_gra_sharp(
    image_size=(160,72,72), patches=(4,4,4), dim=96, ff_dim=144,
    num_heads=4, num_layers=12, dropout_rate=0.1, theta=0.7)
print(f'FactorizePhys_FSAM_Res params: {sum(p.numel() for p in _fp.parameters()):,}')
print(f'PhysFormer params            : {sum(p.numel() for p in _pf.parameters()):,}')
del _fp, _pf

FactorizePhys_FSAM_Res params: 52,169
PhysFormer params            : 7,380,871


In [5]:
# Cell 5 — Generic inference runners

def run_clips_inference(model, clips_tensor, arch='fp', batch_size=4, device='cuda:0'):
    """
    Run inference on a [N, 3, 161, 72, 72] float32 tensor.
    Returns list of per-clip HR (bpm).
    """
    pred_hrs = []
    N = clips_tensor.shape[0]
    with torch.no_grad():
        for start in range(0, N, batch_size):
            batch = clips_tensor[start:start + batch_size].to(device)
            if arch == 'fp':
                out   = model(batch)
                preds = out[0].float().cpu().numpy()
            else:
                out   = model(batch, gra_sharp=2.0)
                preds = out[0].float().cpu().numpy() if isinstance(out, tuple) else out.float().cpu().numpy()
            for p in preds:
                pred_hrs.append(extract_hr(p, fps=FPS))
    return pred_hrs


def run_loader_inference(model, dataset, arch='fp', batch_size=4, device='cuda:0'):
    """
    Run inference over a Dataset (streaming, avoids pre-stacking large tensors).
    Dataset __getitem__ must return a dict with key 'frames' → [3, 161, H, W] tensor.
    Returns list of per-clip HR (bpm).
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=0, pin_memory=False)
    pred_hrs = []
    with torch.no_grad():
        for batch in tqdm(loader, leave=False):
            frames = batch['frames'].to(device)
            if arch == 'fp':
                out   = model(frames)
                preds = out[0].float().cpu().numpy()
            else:
                out   = model(frames, gra_sharp=2.0)
                preds = out[0].float().cpu().numpy() if isinstance(out, tuple) else out.float().cpu().numpy()
            for p in preds:
                pred_hrs.append(extract_hr(p, fps=FPS))
    return pred_hrs


print('Inference runners ready.')

Inference runners ready.


In [6]:
# Cell 6 — Dataset A: UBFC-rPPG
# FactorizePhys results copied directly from notebook 06 (do not re-run).
# PhysFormer: run fresh using the UBFC cache.

class UBFCCacheDataset(Dataset):
    def __init__(self, cache_path):
        self.clips = torch.load(str(cache_path), weights_only=False)
    def __len__(self):
        return len(self.clips)
    def __getitem__(self, idx):
        return self.clips[idx]


ubfc_ds       = UBFCCacheDataset(UBFC_CACHE)
ubfc_subjects = [c['subj'] for c in ubfc_ds.clips]
ubfc_unique   = sorted(set(ubfc_subjects), key=lambda s: int(s.replace('subject', '')))
ubfc_gt_hrs   = defaultdict(list)

# Pre-collect GT HR per subject from cache PPG signals (same for all model runs)
for clip, subj in zip(ubfc_ds.clips, ubfc_subjects):
    ubfc_gt_hrs[subj].append(extract_hr(clip['ppg'].numpy(), fps=FPS))
ubfc_gt_subj = np.array([np.mean(ubfc_gt_hrs[s]) for s in ubfc_unique])

print(f'UBFC clips      : {len(ubfc_ds)} from {len(ubfc_unique)} subjects')
print(f'GT HR range     : {ubfc_gt_subj.min():.1f} – {ubfc_gt_subj.max():.1f} bpm')
print('NOTE: UBFC clips are NOT pre-stacked into a tensor — inference streams from Dataset to avoid 4.8 GB allocation.')

# ── FactorizePhys UBFC results from notebook 06 (verbatim) ───────────────────
# Run on 2026-05-23. Protocol: clip-level FFT-peak HR averaged per subject.
FP_UBFC_RESULTS = [
    {'name': 'FP_SCAMPS',        'MAE': 1.06, 'RMSE': 1.91, 'Pearson_r': 0.9933},
    {'name': 'FP_PURE',          'MAE': 1.23, 'RMSE': 2.05, 'Pearson_r': 0.9923},
    {'name': 'FP_iBVP',          'MAE': 0.81, 'RMSE': 1.23, 'Pearson_r': 0.9969},
    {'name': 'FP_Soup_S+P',      'MAE': 0.94, 'RMSE': 1.66, 'Pearson_r': 0.9946},
    {'name': 'FP_Soup_S+I',      'MAE': 0.94, 'RMSE': 1.56, 'Pearson_r': 0.9950},
    {'name': 'FP_Soup_S+P+I',    'MAE': 0.92, 'RMSE': 1.56, 'Pearson_r': 0.9950},
]
print('\nFactorizePhys UBFC results (from notebook 06):')
for r in FP_UBFC_RESULTS:
    print(f'  {r["name"]:20s}  MAE={r["MAE"]:.2f}  RMSE={r["RMSE"]:.2f}  r={r["Pearson_r"]:.4f}')

UBFC clips      : 483 from 42 subjects
GT HR range     : 60.0 – 128.4 bpm
NOTE: UBFC clips are NOT pre-stacked into a tensor — inference streams from Dataset to avoid 4.8 GB allocation.

FactorizePhys UBFC results (from notebook 06):
  FP_SCAMPS             MAE=1.06  RMSE=1.91  r=0.9933
  FP_PURE               MAE=1.23  RMSE=2.05  r=0.9923
  FP_iBVP               MAE=0.81  RMSE=1.23  r=0.9969
  FP_Soup_S+P           MAE=0.94  RMSE=1.66  r=0.9946
  FP_Soup_S+I           MAE=0.94  RMSE=1.56  r=0.9950
  FP_Soup_S+P+I         MAE=0.92  RMSE=1.56  r=0.9950


In [7]:
# Cell 7 — UBFC: run PhysFormer (4 configs)
# Uses run_loader_inference (streaming from Dataset, no 4.8 GB tensor in RAM).

PF_UBFC_CONFIGS = [
    ('PF_SCAMPS',     [CKPT_PF_SCAMPS]),
    ('PF_PURE',       [CKPT_PF_PURE]),
    ('PF_iBVP',       [CKPT_PF_IBVP]),
    ('PF_Soup_S+P+I', [CKPT_PF_SCAMPS, CKPT_PF_PURE, CKPT_PF_IBVP]),
]

PF_UBFC_RESULTS = []

for name, ckpt_paths in PF_UBFC_CONFIGS:
    print(f'\n{"="*55}')
    print(f'UBFC  PhysFormer: {name}')
    t0 = time.time()

    sd    = load_stripped(ckpt_paths[0]) if len(ckpt_paths) == 1 else make_soup(ckpt_paths)
    model = build_physformer(sd, device)

    clip_pred_hrs = run_loader_inference(
        model, ubfc_ds, arch='pf', batch_size=BATCH_SIZE, device=device
    )

    # Aggregate per-subject
    subj_pred = defaultdict(list)
    for subj, ph in zip(ubfc_subjects, clip_pred_hrs):
        subj_pred[subj].append(ph)
    pred_subj = np.array([np.mean(subj_pred[s]) for s in ubfc_unique])

    metrics = compute_metrics(pred_subj, ubfc_gt_subj)
    elapsed = time.time() - t0
    print(f'  HR MAE    : {metrics["MAE"]:.2f} bpm')
    print(f'  HR RMSE   : {metrics["RMSE"]:.2f} bpm')
    print(f'  Pearson r : {metrics["Pearson_r"]:.4f}')
    print(f'  Time      : {elapsed:.1f}s')

    PF_UBFC_RESULTS.append({
        'name':      name,
        'MAE':       round(metrics['MAE'],       2),
        'RMSE':      round(metrics['RMSE'],      2),
        'Pearson_r': round(metrics['Pearson_r'], 4),
    })

    del model, sd
    torch.cuda.empty_cache()
    gc.collect()

print('\nPhysFormer UBFC evaluation complete.')


UBFC  PhysFormer: PF_SCAMPS


  0%|          | 0/121 [00:00<?, ?it/s]

  HR MAE    : 44.68 bpm
  HR RMSE   : 48.60 bpm
  Pearson r : 0.1010
  Time      : 23.0s



UBFC  PhysFormer: PF_PURE


  0%|          | 0/121 [00:00<?, ?it/s]

...

2026-05-23 19:46:24,570 - clearml.model - WARNING - Connecting multiple input models with the same name: `SCAMPS_PhysFormer`. This might result in the wrong model being used when executing remotely


2026-05-23 19:46:30,212 - clearml.model - WARNING - Connecting multiple input models with the same name: `PURE_PhysFormer`. This might result in the wrong model being used when executing remotely


2026-05-23 19:46:35,839 - clearml.model - WARNING - Connecting multiple input models with the same name: `iBVP_PhysFormer`. This might result in the wrong model being used when executing remotely


  0%|          | 0/121 [00:00<?, ?it/s]

  HR MAE    : 55.29 bpm
  HR RMSE   : 57.39 bpm
  Pearson r : 0.2628
  Time      : 32.6s

PhysFormer UBFC evaluation complete.


In [8]:
# Cell 8 — Dataset B: rPPG-10 — ECG GT extraction only (no frame pre-loading)
# Load frames on-demand per subject in Cell 10 to avoid OOM (~28 GB if pre-cached).
# GT HR from ECG via R-peak detection.
from scipy.signal import find_peaks

def extract_hr_ecg(ecg_signal, ecg_fs=1000.0):
    """R-peak detection (Pan-Tompkins style). FFT unreliable for non-sinusoidal ECG."""
    nyq = ecg_fs / 2.0
    b, a = butter(4, [5.0/nyq, 20.0/nyq], btype='bandpass')
    filtered = filtfilt(b, a, ecg_signal.astype('float64'))
    squared  = filtered ** 2
    min_dist = int(0.3 * ecg_fs)
    peaks, _ = find_peaks(squared, distance=min_dist, height=np.percentile(squared, 75))
    if len(peaks) < 2:
        return 75.0
    rr = np.diff(peaks) / ecg_fs
    rr = rr[(rr >= 0.3) & (rr <= 2.0)]
    return float(60.0 / np.mean(rr)) if len(rr) > 0 else 75.0


def load_rppg10_clips(subj_dir, clip_len=160, target_size=72):
    """Load one subject's Cheek1 video frames and chunk into clips. Returns tensor or None."""
    subj_name  = subj_dir.name
    video_path = subj_dir / f'{subj_name}_Cheek1_.avi'
    if not video_path.exists():
        return None
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        frames.append(cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB),
                                  (target_size, target_size), interpolation=cv2.INTER_AREA))
    cap.release()
    if len(frames) < clip_len + 1:
        return None
    frames_np = np.stack(frames).astype('float32') / 255.0  # [T,H,W,3]
    frames_np = frames_np.transpose(0, 3, 1, 2)              # [T,3,H,W]
    n = frames_np.shape[0] // (clip_len + 1)
    if n == 0: return None
    clips = [torch.from_numpy(frames_np[i*(clip_len+1):(i+1)*(clip_len+1)]) for i in range(n)]
    clips_t = torch.stack(clips).permute(0, 2, 1, 3, 4)     # [N,3,161,H,W]
    return clips_t


# Build metadata dict: subject_name -> (video_dir, gt_hr)
print('Building rPPG-10 metadata (ECG GT only, no frame loading) ...')
rppg10_meta = {}  # subj_name -> (subj_dir, gt_hr)
for sid in range(1, 28):
    subj_dir = RPPG10_ROOT / f'Subject_{sid}'
    if not subj_dir.exists():
        print(f'  [SKIP] Subject_{sid}: directory missing')
        continue
    ecg_path = subj_dir / f'Subject_{sid}_ECG.npy'
    video_path = subj_dir / f'Subject_{sid}_Cheek1_.avi'
    if not ecg_path.exists() or not video_path.exists():
        print(f'  [SKIP] Subject_{sid}: missing ECG or Cheek1 video')
        continue
    try:
        ecg   = np.load(str(ecg_path)).astype('float64')
        gt_hr = extract_hr_ecg(ecg, ecg_fs=1000.0)
    except Exception as ex:
        print(f'  [SKIP] Subject_{sid}: ECG load failed ({ex})')
        continue
    rppg10_meta[f'Subject_{sid}'] = (subj_dir, float(gt_hr))
    print(f'  Subject_{sid:2d}: GT HR={gt_hr:.1f} bpm')

rppg10_subjects = sorted(rppg10_meta.keys(), key=lambda s: int(s.split('_')[1]))
rppg10_gt       = np.array([rppg10_meta[s][1] for s in rppg10_subjects])
print(f'\nrPPG-10 subjects: {len(rppg10_subjects)}')
print(f'GT HR range: {rppg10_gt.min():.1f} - {rppg10_gt.max():.1f} bpm')
print('NOTE: frames loaded on-demand per subject during inference to avoid OOM.')


Building rPPG-10 metadata (ECG GT only, no frame loading) ...


  Subject_ 1: GT HR=102.2 bpm
  Subject_ 2: GT HR=93.8 bpm
  Subject_ 3: GT HR=142.4 bpm
  Subject_ 4: GT HR=94.8 bpm
  Subject_ 5: GT HR=81.7 bpm
  Subject_ 6: GT HR=92.5 bpm
  Subject_ 7: GT HR=76.0 bpm
  Subject_ 8: GT HR=112.1 bpm
  Subject_ 9: GT HR=90.5 bpm


  Subject_10: GT HR=75.3 bpm


  Subject_11: GT HR=86.5 bpm
  Subject_12: GT HR=106.1 bpm
  Subject_13: GT HR=67.9 bpm
  Subject_14: GT HR=111.8 bpm
  Subject_15: GT HR=79.8 bpm
  Subject_16: GT HR=88.2 bpm
  Subject_17: GT HR=118.1 bpm
  Subject_18: GT HR=86.0 bpm
  Subject_19: GT HR=77.7 bpm


  Subject_20: GT HR=68.8 bpm


  Subject_21: GT HR=100.7 bpm
  Subject_22: GT HR=102.6 bpm
  Subject_23: GT HR=87.4 bpm
  Subject_24: GT HR=84.6 bpm
  Subject_25: GT HR=121.0 bpm
  Subject_26: GT HR=67.2 bpm
  Subject_27: GT HR=81.6 bpm

rPPG-10 subjects: 27
GT HR range: 67.2 - 142.4 bpm
NOTE: frames loaded on-demand per subject during inference to avoid OOM.


In [9]:
# Cell 9 — Dataset C: MCD-rPPG — metadata + per-subject lazy loader
# Last 100 patient_ids (sorted), IriunWebcam only, before+after.
# Videos are loaded on-demand during inference (not pre-cached) to stay within RAM.
# GT HR = mean of before+after pulse values from db.csv.

sys.path.insert(0, str(FP_ROOT))
from dataset.data_loader.face_detector.YOLO5Face import YOLO5Face

yolo = YOLO5Face(backend='Y5F', device=str(device))


def load_mcd_video(video_path, yolo_detector, n_frames=960, clip_len=160, target_size=72):
    """
    Load one MCD-rPPG video:
    - Read first n_frames frames.
    - Detect face on frame 0, apply bbox to all frames.
    - Resize crop to target_size x target_size.
    - Chunk into (clip_len+1)-frame clips.
    Returns clips_tensor [N_clips, 3, clip_len+1, H, W] float32 in [0,1], or None.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None

    ret, f0 = cap.read()
    if not ret:
        cap.release()
        return None
    f0_rgb = cv2.cvtColor(f0, cv2.COLOR_BGR2RGB)
    bbox   = yolo_detector.detect_face(f0_rgb)

    if bbox is None:
        h, w = f0.shape[:2]
        bbox = [0, 0, w, h]
    x1, y1, x2, y2 = [int(v) for v in bbox]
    x1, y1 = max(0, x1), max(0, y1)

    def crop_resize(frame_bgr):
        rgb  = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        crop = rgb[y1:y2, x1:x2]
        if crop.size == 0:
            crop = rgb
        return cv2.resize(crop, (target_size, target_size), interpolation=cv2.INTER_AREA)

    frames = [crop_resize(f0)]
    while len(frames) < n_frames:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(crop_resize(frame))
    cap.release()

    if len(frames) < clip_len + 1:
        return None

    frames_np = np.stack(frames[:n_frames]).astype('float32') / 255.0  # [T, H, W, 3]
    frames_np = frames_np.transpose(0, 3, 1, 2)                        # [T, 3, H, W]

    n_clips = frames_np.shape[0] // (clip_len + 1)
    if n_clips == 0:
        return None

    clips = []
    for i in range(n_clips):
        s = i * (clip_len + 1)
        clips.append(torch.from_numpy(frames_np[s:s + clip_len + 1]))

    clips_t = torch.stack(clips)            # [N_clips, clip_len+1, 3, H, W]
    clips_t = clips_t.permute(0, 2, 1, 3, 4)  # [N_clips, 3, clip_len+1, H, W]
    return clips_t


# ── Load MCD-rPPG metadata only (videos loaded on-demand in Cell 10) ──────────
mcd_df   = pd.read_csv(str(MCD_DB_CSV))
all_pids = sorted(mcd_df['patient_id'].unique())
last100  = all_pids[-100:]

mcd_sub  = mcd_df[
    (mcd_df['patient_id'].isin(last100)) & (mcd_df['camera'] == 'IriunWebcam')
].copy()
mcd_sub  = mcd_sub.sort_values(['patient_id', 'step'])

# Build per-pid GT HR dict
mcd_gt_dict = {}
mcd_pids    = []
skip_meta   = 0
for pid in last100:
    rows = mcd_sub[mcd_sub['patient_id'] == pid]
    if len(rows) != 2:
        skip_meta += 1
        continue
    before_row = rows[rows['step'] == 'before'].iloc[0]
    after_row  = rows[rows['step'] == 'after'].iloc[0]
    gt_hr = float((before_row['pulse'] + after_row['pulse']) / 2.0)
    # Verify video files exist
    vb = MCD_VIDEO_DIR / f'{pid}_IriunWebcam_before.avi'
    va = MCD_VIDEO_DIR / f'{pid}_IriunWebcam_after.avi'
    if not vb.exists() or not va.exists():
        skip_meta += 1
        continue
    mcd_pids.append(pid)
    mcd_gt_dict[pid] = gt_hr

mcd_gt_hrs = np.array([mcd_gt_dict[p] for p in mcd_pids])
print(f'MCD-rPPG subset  : {len(mcd_pids)} subjects ({skip_meta} skipped at metadata stage)')
print(f'GT HR range      : {mcd_gt_hrs.min():.1f} – {mcd_gt_hrs.max():.1f} bpm')
print('Videos will be loaded on-demand per subject during inference (memory-safe).')

I0000 00:00:1779529617.575793 4165697 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


MCD-rPPG subset  : 100 subjects (0 skipped at metadata stage)
GT HR range      : 62.0 – 134.0 bpm
Videos will be loaded on-demand per subject during inference (memory-safe).


In [10]:
# Cell 10 — Evaluation loop: all 10 configs × all 3 datasets
# rPPG-10 data was pre-loaded in Cell 8 (rppg10_data, rppg10_subjects, rppg10_gt).
# MCD-rPPG is loaded per-subject on demand (memory-safe for 12 GB dataset).

# ── 10-config × 3-dataset loop ───────────────────────────────────────────────
ALL_CONFIGS = [
    ('FP_SCAMPS',     'fp', [CKPT_FP_SCAMPS]),
    ('FP_PURE',       'fp', [CKPT_FP_PURE]),
    ('FP_iBVP',       'fp', [CKPT_FP_IBVP]),
    ('FP_Soup_S+P',   'fp', [CKPT_FP_SCAMPS, CKPT_FP_PURE]),
    ('FP_Soup_S+I',   'fp', [CKPT_FP_SCAMPS, CKPT_FP_IBVP]),
    ('FP_Soup_S+P+I', 'fp', [CKPT_FP_SCAMPS, CKPT_FP_PURE, CKPT_FP_IBVP]),
    ('PF_SCAMPS',     'pf', [CKPT_PF_SCAMPS]),
    ('PF_PURE',       'pf', [CKPT_PF_PURE]),
    ('PF_iBVP',       'pf', [CKPT_PF_IBVP]),
    ('PF_Soup_S+P+I', 'pf', [CKPT_PF_SCAMPS, CKPT_PF_PURE, CKPT_PF_IBVP]),
]

UBFC_LOOKUP = {r['name']: r for r in FP_UBFC_RESULTS}
UBFC_LOOKUP.update({r['name']: r for r in PF_UBFC_RESULTS})

ALL_RESULTS = []

for name, arch, ckpt_paths in ALL_CONFIGS:
    print(f'\n{"="*60}')
    print(f'Config: {name}  arch={arch}')
    t0 = time.time()

    # UBFC — look up pre-computed results (no re-run)
    ubfc_r = UBFC_LOOKUP.get(name)
    if ubfc_r is None:
        print('  [WARNING] UBFC result missing in lookup')
        ubfc_mae, ubfc_rmse, ubfc_r_val = float('nan'), float('nan'), float('nan')
    else:
        ubfc_mae, ubfc_rmse, ubfc_r_val = ubfc_r['MAE'], ubfc_r['RMSE'], ubfc_r['Pearson_r']
    print(f'  UBFC   MAE={ubfc_mae:.2f}  RMSE={ubfc_rmse:.2f}  r={ubfc_r_val:.4f}')

    # Load model
    sd    = load_stripped(ckpt_paths[0]) if len(ckpt_paths) == 1 else make_soup(ckpt_paths)
    model = build_factorizephys(sd, device) if arch == 'fp' else build_physformer(sd, device)

    # ── rPPG-10 (load per subject on demand to avoid OOM) ───────────────────
    rppg10_pred = []
    for subj in tqdm(rppg10_subjects, desc=f'rPPG-10 {name}', leave=False):
        subj_dir, _ = rppg10_meta[subj]
        clips_t = load_rppg10_clips(subj_dir, clip_len=RPPG10_CLIPS, target_size=72)
        if clips_t is None:
            rppg10_pred.append(75.0)
            continue
        clip_hrs = run_clips_inference(
            model, clips_t, arch=arch, batch_size=BATCH_SIZE, device=device
        )
        rppg10_pred.append(float(np.mean(clip_hrs)))

    rppg10_pred = np.array(rppg10_pred)
    r10_m = compute_metrics(rppg10_pred, rppg10_gt)
    print(f'  rPPG10 MAE={r10_m["MAE"]:.2f}  RMSE={r10_m["RMSE"]:.2f}  r={r10_m["Pearson_r"]:.4f}')

    # ── MCD-rPPG (load each subject's videos on demand) ───────────────────────
    mcd_pred   = []
    mcd_skip   = 0
    for pid in tqdm(mcd_pids, desc=f'MCD {name}', leave=False):
        clips_b = load_mcd_video(
            MCD_VIDEO_DIR / f'{pid}_IriunWebcam_before.avi',
            yolo, n_frames=MCD_N_FRAMES, clip_len=MCD_CLIP_LEN
        )
        clips_a = load_mcd_video(
            MCD_VIDEO_DIR / f'{pid}_IriunWebcam_after.avi',
            yolo, n_frames=MCD_N_FRAMES, clip_len=MCD_CLIP_LEN
        )
        if clips_b is None or clips_a is None:
            mcd_pred.append(float('nan'))
            mcd_skip += 1
            continue

        hrs_b = run_clips_inference(model, clips_b, arch=arch, batch_size=BATCH_SIZE, device=device)
        hrs_a = run_clips_inference(model, clips_a, arch=arch, batch_size=BATCH_SIZE, device=device)
        mcd_pred.append((float(np.mean(hrs_b)) + float(np.mean(hrs_a))) / 2.0)

    mcd_pred_arr = np.array(mcd_pred)
    valid_mask   = ~np.isnan(mcd_pred_arr)
    mcd_m = compute_metrics(mcd_pred_arr[valid_mask], mcd_gt_hrs[valid_mask])
    if mcd_skip > 0:
        print(f'  MCD skip: {mcd_skip} subjects with load errors')
    print(f'  MCD    MAE={mcd_m["MAE"]:.2f}  RMSE={mcd_m["RMSE"]:.2f}  r={mcd_m["Pearson_r"]:.4f}')
    print(f'  Total time: {time.time()-t0:.1f}s')

    ALL_RESULTS.append({
        'name':      name,
        'arch':      arch,
        'ubfc_mae':  round(ubfc_mae,  2),
        'ubfc_rmse': round(ubfc_rmse, 2),
        'ubfc_r':    round(ubfc_r_val,4),
        'r10_mae':   round(r10_m['MAE'],       2),
        'r10_rmse':  round(r10_m['RMSE'],      2),
        'r10_r':     round(r10_m['Pearson_r'], 4),
        'mcd_mae':   round(mcd_m['MAE'],       2),
        'mcd_rmse':  round(mcd_m['RMSE'],      2),
        'mcd_r':     round(mcd_m['Pearson_r'], 4),
        'r10_pred':  rppg10_pred,
        'r10_gt':    rppg10_gt,
        'mcd_pred':  mcd_pred_arr[valid_mask],
        'mcd_gt':    mcd_gt_hrs[valid_mask],
    })

    del model, sd
    torch.cuda.empty_cache()
    gc.collect()

print('\nAll 10 configs evaluated across all 3 datasets.')


Config: FP_SCAMPS  arch=fp
  UBFC   MAE=1.06  RMSE=1.91  r=0.9933


rPPG-10 FP_SCAMPS:   0%|          | 0/27 [00:00<?, ?it/s]

  rPPG10 MAE=17.72  RMSE=20.73  r=-0.1332


MCD FP_SCAMPS:   0%|          | 0/100 [00:00<?, ?it/s]

  MCD    MAE=14.38  RMSE=17.58  r=0.2698
  Total time: 128.7s


...

rPPG-10 PF_Soup_S+P+I:   0%|          | 0/27 [00:00<?, ?it/s]

  rPPG10 MAE=46.38  RMSE=49.87  r=0.0253


MCD PF_Soup_S+P+I:   0%|          | 0/100 [00:00<?, ?it/s]

  MCD    MAE=47.31  RMSE=49.37  r=-0.0274
  Total time: 218.0s



All 10 configs evaluated across all 3 datasets.


In [11]:
# Cell 11 — Summary tables and plots

# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for r in ALL_RESULTS:
    rows.append({
        'Model':       r['name'],
        'UBFC MAE':    r['ubfc_mae'],
        'UBFC RMSE':   r['ubfc_rmse'],
        'UBFC r':      r['ubfc_r'],
        'rPPG10 MAE':  r['r10_mae'],
        'rPPG10 RMSE': r['r10_rmse'],
        'rPPG10 r':    r['r10_r'],
        'MCD MAE':     r['mcd_mae'],
        'MCD RMSE':    r['mcd_rmse'],
        'MCD r':       r['mcd_r'],
    })

df_summary = pd.DataFrame(rows)
print('\n' + '='*115)
print('FULL CROSS-DATASET BENCHMARK — HR-level Pearson r (across subjects, NOT waveform-level r)')
print('='*115)
print(df_summary.to_string(index=False))
print('='*115)

# ── MAE bar chart: all models × all datasets ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)
datasets  = [('UBFC-rPPG', 'ubfc_mae'), ('rPPG-10', 'r10_mae'), ('MCD-rPPG', 'mcd_mae')]
colors    = ['#4878cf'] * 6 + ['#e07b39'] * 4
names     = [r['name'] for r in ALL_RESULTS]

for ax, (ds_name, mae_key) in zip(axes, datasets):
    maes = [r[mae_key] for r in ALL_RESULTS]
    bars = ax.bar(range(len(names)), maes, color=colors,
                  alpha=0.85, edgecolor='black', linewidth=0.4)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=35, ha='right', fontsize=8)
    ax.set_ylabel('HR MAE (bpm)', fontsize=9)
    ax.set_title(f'{ds_name}', fontsize=11)
    ax.set_ylim(0, max(maes) * 1.35 + 0.5)
    for bar, mae in zip(bars, maes):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{mae:.2f}', ha='center', va='bottom', fontsize=7)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4878cf', label='FactorizePhys'),
    Patch(facecolor='#e07b39', label='PhysFormer'),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, 1.02))
plt.suptitle('HR MAE (bpm) — Cross-Dataset Benchmark', fontsize=13, y=1.05)
plt.tight_layout()
bar_path = PROJECT_ROOT / 'docs' / 'benchmark_mae_bar.png'
plt.savefig(str(bar_path), dpi=150, bbox_inches='tight')
plt.show()
print(f'Bar chart saved -> {bar_path}')

# ── Scatter plots: best FP and best PF on rPPG-10 and MCD ────────────────────
def mean_mae(r):
    return (r['ubfc_mae'] + r['r10_mae'] + r['mcd_mae']) / 3.0

best_fp = min([r for r in ALL_RESULTS if r['arch'] == 'fp'], key=mean_mae)
best_pf = min([r for r in ALL_RESULTS if r['arch'] == 'pf'], key=mean_mae)

fig2, axes2 = plt.subplots(2, 2, figsize=(12, 9))
plot_items = [
    (axes2[0][0], best_fp, 'r10_pred', 'r10_gt',  'r10_mae', 'r10_r', 'rPPG-10'),
    (axes2[0][1], best_fp, 'mcd_pred', 'mcd_gt',  'mcd_mae', 'mcd_r', 'MCD-rPPG'),
    (axes2[1][0], best_pf, 'r10_pred', 'r10_gt',  'r10_mae', 'r10_r', 'rPPG-10'),
    (axes2[1][1], best_pf, 'mcd_pred', 'mcd_gt',  'mcd_mae', 'mcd_r', 'MCD-rPPG'),
]

for ax, r, pred_k, gt_k, mae_k, r_k, ds in plot_items:
    pred = r[pred_k]
    gt   = r[gt_k]
    ax.scatter(gt, pred, alpha=0.7, s=30, color='steelblue', edgecolors='white', linewidths=0.4)
    lim = [min(gt.min(), pred.min()) - 5, max(gt.max(), pred.max()) + 5]
    ax.plot(lim, lim, 'r--', lw=1)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel('GT HR (bpm)', fontsize=9)
    ax.set_ylabel('Pred HR (bpm)', fontsize=9)
    ax.set_title(f'{r["name"]} | {ds}\nMAE={r[mae_k]:.2f} bpm  r={r[r_k]:.3f}', fontsize=9)
    ax.tick_params(labelsize=8)

plt.suptitle(
    f'Best FP: {best_fp["name"]} (mean MAE {mean_mae(best_fp):.2f})  |  '
    f'Best PF: {best_pf["name"]} (mean MAE {mean_mae(best_pf):.2f})\n'
    f'HR-level scatter (not waveform-level)',
    fontsize=10
)
plt.tight_layout()
scatter_path = PROJECT_ROOT / 'docs' / 'benchmark_scatter.png'
plt.savefig(str(scatter_path), dpi=150, bbox_inches='tight')
plt.show()
print(f'Scatter plot saved -> {scatter_path}')


FULL CROSS-DATASET BENCHMARK — HR-level Pearson r (across subjects, NOT waveform-level r)
        Model  UBFC MAE  UBFC RMSE  UBFC r  rPPG10 MAE  rPPG10 RMSE  rPPG10 r  MCD MAE  MCD RMSE   MCD r
    FP_SCAMPS      1.06       1.91  0.9933       17.72        20.73   -0.1332    14.38     17.58  0.2698
      FP_PURE      1.23       2.05  0.9923       13.89        18.61    0.1766    12.36     16.13  0.0705
      FP_iBVP      0.81       1.23  0.9969       18.17        23.55   -0.0744    14.35     19.16 -0.1258
  FP_Soup_S+P      0.94       1.66  0.9946       16.70        19.50    0.0267    13.76     16.25  0.1984
  FP_Soup_S+I      0.94       1.56  0.9950       15.75        20.95    0.2750    13.77     18.31 -0.1214
FP_Soup_S+P+I      0.92       1.56  0.9950       14.34        19.03   -0.0114    12.52     16.25  0.0738
    PF_SCAMPS     44.68      48.60  0.1010       16.15        22.16   -0.1276    39.80     43.44 -0.1319
      PF_PURE     46.67      49.84 -0.3902       31.31        36.02  

Bar chart saved -> /mnt/sata-ssd/rppg_project/docs/benchmark_mae_bar.png


Scatter plot saved -> /mnt/sata-ssd/rppg_project/docs/benchmark_scatter.png


In [12]:
# Cell 12 — Save best overall checkpoint

# mean_mae defined in Cell 11; best_fp and best_pf also defined there.
best_overall = min(ALL_RESULTS, key=mean_mae)
print(f'Best overall: {best_overall["name"]}')
print(f'  UBFC  MAE={best_overall["ubfc_mae"]:.2f}  RMSE={best_overall["ubfc_rmse"]:.2f}  r={best_overall["ubfc_r"]:.4f}')
print(f'  rPPG10 MAE={best_overall["r10_mae"]:.2f}  RMSE={best_overall["r10_rmse"]:.2f}  r={best_overall["r10_r"]:.4f}')
print(f'  MCD   MAE={best_overall["mcd_mae"]:.2f}  RMSE={best_overall["mcd_rmse"]:.2f}  r={best_overall["mcd_r"]:.4f}')
print(f'  Mean  MAE={mean_mae(best_overall):.2f}')

config_map   = {n: (arch, ckpts) for n, arch, ckpts in ALL_CONFIGS}
best_arch, best_ckpts = config_map[best_overall['name']]
best_sd = load_stripped(best_ckpts[0]) if len(best_ckpts) == 1 else make_soup(best_ckpts)

best_pth = BEST_CKPT_DIR / 'best.pth'
torch.save(best_sd, str(best_pth))
print(f'Best checkpoint saved -> {best_pth}')

report = {
    'date':           '2026-05-23',
    'best_model':     best_overall['name'],
    'architecture':   best_arch,
    'checkpoints':    [str(p) for p in best_ckpts],
    'mean_mae_bpm':   round(mean_mae(best_overall), 2),
    'metrics': {
        'UBFC_rPPG': {'MAE': best_overall['ubfc_mae'], 'RMSE': best_overall['ubfc_rmse'],
                      'Pearson_r_HR': best_overall['ubfc_r']},
        'rPPG_10':   {'MAE': best_overall['r10_mae'],  'RMSE': best_overall['r10_rmse'],
                      'Pearson_r_HR': best_overall['r10_r']},
        'MCD_rPPG':  {'MAE': best_overall['mcd_mae'],  'RMSE': best_overall['mcd_rmse'],
                      'Pearson_r_HR': best_overall['mcd_r']},
    },
    'all_results': [
        {
            'name': r['name'],
            'ubfc_mae': r['ubfc_mae'], 'ubfc_rmse': r['ubfc_rmse'], 'ubfc_r': r['ubfc_r'],
            'r10_mae':  r['r10_mae'],  'r10_rmse':  r['r10_rmse'],  'r10_r':  r['r10_r'],
            'mcd_mae':  r['mcd_mae'],  'mcd_rmse':  r['mcd_rmse'],  'mcd_r':  r['mcd_r'],
        }
        for r in ALL_RESULTS
    ],
    'saved_checkpoint': str(best_pth),
    'note': 'Pearson r is HR-level (across subjects), not waveform-level.',
}

report_path = BEST_CKPT_DIR / 'best_report.json'
with open(str(report_path), 'w') as f:
    json.dump(report, f, indent=2)
print(f'JSON report saved  -> {report_path}')

Best overall: FP_PURE
  UBFC  MAE=1.23  RMSE=2.05  r=0.9923
  rPPG10 MAE=13.89  RMSE=18.61  r=0.1766
  MCD   MAE=12.36  RMSE=16.13  r=0.0705
  Mean  MAE=9.16


Best checkpoint saved -> /mnt/sata-ssd/rppg_project/checkpoints/best_pretrained/best.pth
JSON report saved  -> /mnt/sata-ssd/rppg_project/checkpoints/best_pretrained/best_report.json


In [13]:
# Cell 13 — ClearML metric logging

if logger is not None:
    for r in ALL_RESULTS:
        tag = r['name']
        logger.report_scalar('UBFC_MAE_bpm',    tag, r['ubfc_mae'],  iteration=0)
        logger.report_scalar('UBFC_RMSE_bpm',   tag, r['ubfc_rmse'], iteration=0)
        logger.report_scalar('UBFC_Pearson_r',  tag, r['ubfc_r'],    iteration=0)
        logger.report_scalar('rPPG10_MAE_bpm',  tag, r['r10_mae'],   iteration=0)
        logger.report_scalar('rPPG10_RMSE_bpm', tag, r['r10_rmse'],  iteration=0)
        logger.report_scalar('rPPG10_Pearson_r',tag, r['r10_r'],     iteration=0)
        logger.report_scalar('MCD_MAE_bpm',     tag, r['mcd_mae'],   iteration=0)
        logger.report_scalar('MCD_RMSE_bpm',    tag, r['mcd_rmse'],  iteration=0)
        logger.report_scalar('MCD_Pearson_r',   tag, r['mcd_r'],     iteration=0)
        logger.report_scalar('Mean_MAE_bpm',    tag, round(mean_mae(r), 2), iteration=0)

    task.upload_artifact('benchmark_mae_bar',   artifact_object=str(bar_path))
    task.upload_artifact('benchmark_scatter',   artifact_object=str(scatter_path))
    task.upload_artifact('best_report_json',    artifact_object=str(report_path))
    task.close()
    print('ClearML metrics and artifacts logged. Task closed.')
else:
    print('ClearML not available — metrics recorded locally only.')

ClearML metrics and artifacts logged. Task closed.


## Cell 14 — Final Report

**Notebook:** `notebooks/07_full_benchmark.ipynb`  
**Date:** 2026-05-23  
**ClearML task:** rPPG / Full Benchmark  
**Device:** cuda:0 (RTX 3060 12 GB)  

### Dataset Details

| Dataset | Subjects | Protocol |
|---|---|---|
| UBFC-rPPG | 42 | Cache `/tmp/ubfc_y5f_clips_72.pt`; clip HR averaged per subject |
| rPPG-10 | up to 27 | Cheek1 video, 72×72, 160-frame clips; GT HR from ECG FFT |
| MCD-rPPG | 100 (last PIDs) | IriunWebcam, first 960 frames, YOLO5Face crop, before+after mean |

### Models

- **FactorizePhys_FSAM_Res** (6 configs): SCAMPS, PURE, iBVP, Soup S+P, Soup S+I, Soup S+P+I  
  UBFC numbers copied from notebook 06 (not re-run).
- **PhysFormer** (ViT_ST_ST_Compact3_TDC_gra_sharp, 4 configs): SCAMPS, PURE, iBVP, Soup S+P+I

### Pearson r Note

All Pearson r values are **HR-level** (predicted HR vs GT HR across subjects).  
This is NOT waveform-level Pearson r. Papers (e.g. FactorizePhys NeurIPS 2024) report waveform-level r.
Our HR-level r values are not directly comparable to those paper values.

### Key Checkpoints

- Best overall: see `checkpoints/best_pretrained/best.pth`  
- JSON report: `checkpoints/best_pretrained/best_report.json`  
- Model soup (best FP soup): `checkpoints/model_soup/best_soup.pth`  

### Full Results

*See Cell 11 output for the complete summary table.*